# [한번 해보기] 음식 리뷰 데이터 활용 유사도 검색


In [44]:
from dotenv import load_dotenv
import numpy as np
from openai import OpenAI
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()
client = OpenAI()

## 1. 데이터 로드


In [37]:
df = pd.read_csv("../data/fine_food_reviews_1k.csv")

# summary: title, text: content
review_df = df[["Summary", "Text"]].rename(
    columns={"Summary": "title", "Text": "content"}
)
review_df["full_content"] = review_df["title"] + " " + review_df["content"]

review_df.head()

,title,content,full_content
0,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,where does one start...and stop... with a tre...
1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Arrived in pieces Not pleased at all. When I o...
2,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...,"It isn't blanc mange, but isn't bad . . . I'm ..."
3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...,These also have SALT and it's not sea salt. I ...
4,Happy with the product,My dog was suffering with itchy skin. He had ...,Happy with the product My dog was suffering wi...


## 2. 임베딩 벡터 변환


In [ ]:
# 임베딩 벡터 변환 함수
def text_to_embedding(texts, model="text-embedding-3-small"):
    texts = [text.replace("\n", " ") for text in texts]
    response = client.embeddings.create(model=model, input=texts)

    return [data.embedding for data in response.data]

In [42]:
embedding = text_to_embedding(review_df["full_content"].tolist())

embedding_df = pd.DataFrame(embedding)
embedding_df.to_csv("../data/review_embedding.csv", index=False)

print(f"{embedding_df.shape = }")

embedding_df.shape = (1000, 1536)


## 3. 코사인 유사도 검색


In [ ]:
review_embedding = pd.read_csv("../data/review_embedding.csv").values
print(f"{review_embedding.shape = }")

review_embedding.shape = (1000, 1536)


In [49]:
def search(query, top_n=5):
    query_embedding = np.array(text_to_embedding([query]))  # 유저 입력 변환
    cosine_sims = cosine_similarity(query_embedding, review_embedding).reshape(
        -1
    )  # 코사인 유사도 계산

    sorted_idx = cosine_sims.argsort()[::-1][:top_n]

    # 유사도 상위 리뷰 출력
    result_df = review_df.iloc[sorted_idx][["title", "content"]].copy()
    result_df["similarity"] = cosine_sims[sorted_idx]

    return result_df

## 4. 검색 실행


In [ ]:
user_input = input("검색어를 입력하세요: ")
results = search(user_input, top_n=5)

for i, (_, row) in enumerate(results.iterrows(), 1):
    print(f'리뷰 [{i}] : 유사도 {row["similarity"]:.4f}')
    print(f'제목: {row["title"]}')
    print(f'내용: {row["content"]}')
    print("-" * 60)

리뷰 [1] : 유사도 0.4207
제목: grocery
내용: I like this grocery.<br /><br />It is too spicy and famous noodle in Korea.<br /><br />I'm not Korean but I can eat it.<br /><br />Customer service is so good because it is so speedy
--------------------------------------------------
리뷰 [2] : 유사도 0.3661
제목: Delicious
내용: Individually packaged assorted rice crackers are really good.  I could without all the packaging.  Really great price for a high quality mix of rice crackers with some wasabi peas included.  I mixed with salted peanuts for a delicious snack.
--------------------------------------------------
리뷰 [3] : 유사도 0.3500
제목: Instant noodle with best taste and texture -- see Recall Info
내용: I just researched about the korean noodle recall because I love this Shin Ramyun noodle -- the best taste and texture instant noodle I have ever tasted. It is the Neoguri Seafood & Mild Noodles and Neoguri Seafood and Spicy Noodles that are problematic, not this Shin Ramyun type by the same company Nongshim,